<a href="https://colab.research.google.com/github/nibaskumar93n-debug/Morphoinformatics/blob/main/ashik_sankey_plot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
from google.colab import files
uploaded = files.upload()

Saving soil Heavy Metals.xlsx to soil Heavy Metals.xlsx


In [36]:
import pandas as pd
import numpy as np

filename = list(uploaded.keys())[0]
df = pd.read_excel(filename) if filename.endswith('.xlsx') else pd.read_csv(filename)
df.columns = df.columns.str.strip()
df.rename(columns={df.columns[0]: 'Sample'}, inplace=True)

metals = ['Ni', 'V', 'Co', 'Cd', 'Fe', 'Mn', 'As', 'Zn', 'Pb', 'Cr', 'Cu']
X = df[metals].values
print("Data shape:", X.shape)
df.head()

Data shape: (40, 11)


,Sample,Ni,V,Co,Cd,Fe,Mn,As,Zn,Pb,Cr,Cu
0,ASA-1,32.1,63.8,51.4,0.170000,27946.15,596.548125,7.902725,84.850000,30.285283,121.113281,51.7
1,ASA-2,38.1,68.8,58.4,1.856698,32341.50,638.338235,7.202664,66.887868,37.443623,149.509688,59.1
2,ASA-3,38.9,73.8,61.1,1.000000,33942.60,492.063158,7.325786,77.447368,45.091421,179.843906,58.3
3,ASA-4,36.5,63.8,55.6,0.060000,28941.75,619.801875,7.873208,72.493750,31.396226,125.585156,57.3
4,ASA-5,36.5,68.8,57.0,1.868774,34847.50,644.476103,7.647799,81.590074,37.826818,150.987891,55.5


In [37]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import zscore
from scipy.spatial.distance import cdist
import numpy as np

scaler = StandardScaler()
Z = scaler.fit_transform(X)

# Full PCA first to get eigenvalues
pca_full = PCA()
pca_full.fit(Z)

eigenvalues = pca_full.explained_variance_
print("Eigenvalues:", np.round(eigenvalues, 3))
print("Variance explained (%):", np.round(pca_full.explained_variance_ratio_ * 100, 1))

n_factors = int(np.sum(eigenvalues > 1))
print(f"\nFactors retained (eigenvalue > 1): {n_factors}")

# PCA with n_factors
pca = PCA(n_components=n_factors)
F = pca.fit_transform(Z)
loadings_raw = pca.components_.T  # (n_metals, n_factors)

# ── Varimax rotation ──────────────────────────────────────────
def varimax(Phi, gamma=1.0, q=20, tol=1e-6):
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        u, s, vh = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma/p) * Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        )
        R = u @ vh
        d = np.sum(s)
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return Phi @ R, R

loadings_rot, R = varimax(loadings_raw)
F_rot = F @ R   # rotated scores

loading_df = pd.DataFrame(loadings_rot, index=metals,
                           columns=[f'PC{i+1}' for i in range(n_factors)])
print("\nVarimax-rotated loading matrix:\n", loading_df.round(3))

Eigenvalues: [5.962e+00 2.664e+00 1.206e+00 8.100e-01 3.550e-01 1.000e-01 7.100e-02
 4.600e-02 4.300e-02 1.900e-02 5.000e-03]
Variance explained (%): [52.8 23.6 10.7  7.2  3.2  0.9  0.6  0.4  0.4  0.2  0. ]

Factors retained (eigenvalue > 1): 3

Varimax-rotated loading matrix:
       PC1    PC2    PC3
Ni  0.321  0.123  0.261
V   0.212  0.087  0.435
Co  0.352  0.257 -0.058
Cd  0.007 -0.112  0.697
Fe  0.393 -0.042  0.112
Mn  0.421 -0.258 -0.056
As  0.405 -0.092 -0.092
Zn  0.343 -0.094 -0.472
Pb -0.026  0.616 -0.018
Cr -0.015  0.622 -0.063
Cu  0.337  0.215  0.051


In [38]:
z_zero = (np.zeros(len(metals)) - scaler.mean_) / scaler.scale_
F_zero = pca.transform(z_zero.reshape(1, -1)) @ R   # rotated zero score

APCS = F_rot - F_zero
print("APCS shape:", APCS.shape)
print("APCS means:", np.round(APCS.mean(axis=0), 4))

APCS shape: (40, 3)
APCS means: [18.5394 13.5333  3.0157]


In [39]:
from sklearn.linear_model import LinearRegression

coeffs = np.zeros((len(metals), n_factors))
intercepts = np.zeros(len(metals))
r2_scores = []

for j, metal in enumerate(metals):
    reg = LinearRegression()
    reg.fit(APCS, X[:, j])
    coeffs[j, :] = reg.coef_
    intercepts[j] = reg.intercept_
    r2_scores.append(reg.score(APCS, X[:, j]))

coeff_df = pd.DataFrame(coeffs, index=metals,
                         columns=[f'Source {i+1}' for i in range(n_factors)])
print("MLR coefficients:\n", coeff_df.round(3))
print("\nR² scores per metal:", dict(zip(metals, np.round(r2_scores, 3))))

MLR coefficients:
     Source 1  Source 2  Source 3
Ni     1.879     0.719     1.524
V      2.207     0.903     4.537
Co     2.113     1.542    -0.351
Cd     0.005    -0.071     0.442
Fe  3352.675  -359.977   953.840
Mn    56.825   -34.929    -7.561
As     0.565    -0.128    -0.128
Zn     4.772    -1.308    -6.560
Pb    -0.119     2.845    -0.084
Cr    -0.260    11.077    -1.123
Cu     1.911     1.220     0.290

R² scores per metal: {'Ni': np.float64(0.947), 'V': np.float64(0.792), 'Co': np.float64(0.928), 'Cd': np.float64(0.69), 'Fe': np.float64(0.9), 'Mn': np.float64(0.948), 'As': np.float64(0.81), 'Zn': np.float64(0.736), 'Pb': np.float64(0.973), 'Cr': np.float64(0.97), 'Cu': np.float64(0.892)}


In [40]:
APCS_means = APCS.mean(axis=0)
metal_means = X.mean(axis=0)

# Raw contribution of source k to metal j
contributions = coeffs * APCS_means   # (n_metals, n_factors)

# Clip negatives to 0 (physically, a source cannot remove a metal)
contributions_pos = np.clip(contributions, 0, None)

# Row-normalize: each metal's contributions sum to 100%
row_sums = contributions_pos.sum(axis=1, keepdims=True)
pct_contrib = np.where(row_sums > 0, contributions_pos / row_sums * 100, 0)

pct_df = pd.DataFrame(pct_contrib, index=metals,
                       columns=[f'Source {i+1}' for i in range(n_factors)])

print("% Contributions (each row sums to 100%):\n", pct_df.round(1))
print("\nRow sums (should all be 100):\n", pct_df.sum(axis=1).round(1))

% Contributions (each row sums to 100%):
     Source 1  Source 2  Source 3
Ni      70.9      19.8       9.4
V       61.2      18.3      20.5
Co      65.2      34.8       0.0
Cd       6.0       0.0      94.0
Fe      95.6       0.0       4.4
Mn     100.0       0.0       0.0
As     100.0       0.0       0.0
Zn     100.0       0.0       0.0
Pb       0.0     100.0       0.0
Cr       0.0     100.0       0.0
Cu      67.1      31.3       1.7

Row sums (should all be 100):
 Ni    100.0
V     100.0
Co    100.0
Cd    100.0
Fe    100.0
Mn    100.0
As    100.0
Zn    100.0
Pb    100.0
Cr    100.0
Cu    100.0
dtype: float64


In [41]:
print("Varimax-rotated loadings (use to identify sources):\n")
print(loading_df.round(3))

# ── Identify dominant metals per PC ──
for col in loading_df.columns:
    dominant = loading_df[col].abs().nlargest(4).index.tolist()
    print(f"{col} dominant metals: {dominant}")

# ── Assign names after inspecting above output ──
# Typical pattern for soil heavy metals:
#   PC with Fe, Mn, V, Co, Ni  → Natural/lithogenic source
#   PC with Cd, Pb, Zn         → Agricultural source
#   PC with Cu, Cr             → Industrial/dumping source
# ADJUST THESE based on your actual loading output above

source_names = {
    'Source 1': 'Natural source',
    'Source 2': 'Agricultural source',
    'Source 3': 'Dumping source',
}

pct_df.rename(columns=source_names, inplace=True)
sources = list(source_names.values())
print("\nFinal % contributions:\n", pct_df.round(1))

Varimax-rotated loadings (use to identify sources):

      PC1    PC2    PC3
Ni  0.321  0.123  0.261
V   0.212  0.087  0.435
Co  0.352  0.257 -0.058
Cd  0.007 -0.112  0.697
Fe  0.393 -0.042  0.112
Mn  0.421 -0.258 -0.056
As  0.405 -0.092 -0.092
Zn  0.343 -0.094 -0.472
Pb -0.026  0.616 -0.018
Cr -0.015  0.622 -0.063
Cu  0.337  0.215  0.051
PC1 dominant metals: ['Mn', 'As', 'Fe', 'Co']
PC2 dominant metals: ['Cr', 'Pb', 'Mn', 'Co']
PC3 dominant metals: ['Cd', 'Zn', 'V', 'Ni']

Final % contributions:
     Natural source  Agricultural source  Dumping source
Ni            70.9                 19.8             9.4
V             61.2                 18.3            20.5
Co            65.2                 34.8             0.0
Cd             6.0                  0.0            94.0
Fe            95.6                  0.0             4.4
Mn           100.0                  0.0             0.0
As           100.0                  0.0             0.0
Zn           100.0                  0.0          

In [42]:
# Inspect loadings to name each source
print("Loading matrix (use this to name sources):\n")
print(loading_df.round(3))

# ── Manually assign source names after inspecting loadings ──
# Example: PC1 loads high on Fe, Mn, V → Natural source
#          PC2 loads high on Cd, Pb, Zn → Agricultural source
#          PC3 loads high on Cu, Cr, Ni → Dumping source
# Adjust these names based on YOUR loading matrix output above

source_names = {
    'Source 1': 'Natural source',
    'Source 2': 'Agricultural source',
    'Source 3': 'Dumping source',
}

pct_df.rename(columns=source_names, inplace=True)
sources = list(source_names.values())
print("\nFinal % contributions:\n", pct_df.round(1))

Loading matrix (use this to name sources):

      PC1    PC2    PC3
Ni  0.321  0.123  0.261
V   0.212  0.087  0.435
Co  0.352  0.257 -0.058
Cd  0.007 -0.112  0.697
Fe  0.393 -0.042  0.112
Mn  0.421 -0.258 -0.056
As  0.405 -0.092 -0.092
Zn  0.343 -0.094 -0.472
Pb -0.026  0.616 -0.018
Cr -0.015  0.622 -0.063
Cu  0.337  0.215  0.051

Final % contributions:
     Natural source  Agricultural source  Dumping source
Ni            70.9                 19.8             9.4
V             61.2                 18.3            20.5
Co            65.2                 34.8             0.0
Cd             6.0                  0.0            94.0
Fe            95.6                  0.0             4.4
Mn           100.0                  0.0             0.0
As           100.0                  0.0             0.0
Zn           100.0                  0.0             0.0
Pb             0.0                100.0             0.0
Cr             0.0                100.0             0.0
Cu            67.1         

In [44]:
import plotly.graph_objects as go

source_colors = {
    'Natural source':      'rgba(24,95,165,0.85)',
    'Agricultural source': 'rgba(99,153,34,0.85)',
    'Dumping source':      'rgba(239,159,39,0.85)',
}
metal_colors = {
    'Ni':'rgba(124,58,237,0.85)', 'V' :'rgba(8,145,178,0.85)',
    'Co':'rgba(5,150,105,0.85)',  'Cd':'rgba(217,119,6,0.85)',
    'Fe':'rgba(220,38,38,0.85)',  'Mn':'rgba(219,39,119,0.85)',
    'As':'rgba(79,70,229,0.85)',  'Zn':'rgba(13,148,136,0.85)',
    'Pb':'rgba(234,88,12,0.85)',  'Cr':'rgba(101,163,13,0.85)',
    'Cu':'rgba(147,51,234,0.85)',
}

all_nodes   = sources + metals
node_colors = [source_colors[s] for s in sources] + \
              [metal_colors[m]  for m in metals]
node_idx    = {n: i for i, n in enumerate(all_nodes)}

link_src, link_tgt, link_val, link_clr = [], [], [], []
for src in sources:
    fade = source_colors[src].replace('0.85', '0.35')
    for metal in metals:
        val = pct_df.loc[metal, src]
        if val > 0:   # only positive contributions
            link_src.append(node_idx[src])
            link_tgt.append(node_idx[metal])
            link_val.append(round(val, 2))
            link_clr.append(fade)

fig = go.Figure(go.Sankey(
    arrangement = 'snap',
    node = dict(
        pad       = 12,
        thickness = 20,
        line      = dict(color='white', width=0.4),
        label     = all_nodes,
        color     = node_colors,
        customdata = all_nodes,
        hovertemplate = '%{customdata}<extra></extra>',
    ),
    link = dict(
        source        = link_src,
        target        = link_tgt,
        value         = link_val,
        color         = link_clr,
        hovertemplate = '%{source.label} → %{target.label}<br>'
                        'Contribution: %{value:.1f}%<extra></extra>',
    ),
))

fig.update_layout(
    title = dict(
        text    = 'APCS-MLR source apportionment of heavy metals',
        font    = dict(size=16, family='Arial'),
        x       = 0.5,
        xanchor = 'center',
    ),
    font          = dict(size=18, family='bold', color='black'),
    paper_bgcolor = 'white',
    height        = 580,
    margin        = dict(l=20, r=20, t=60, b=20),
)

fig.show()

In [48]:
# Save Sankey as interactive HTML
fig.write_html('sankey_APCS_MLR.html')
print("✓ sankey_APCS_MLR.html saved")

# Save Excel results
with pd.ExcelWriter('APCS_MLR_results.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Raw Data', index=False)
    loading_df.round(4).to_excel(writer, sheet_name='PCA Loadings')
    eigen_df = pd.DataFrame({
        'Component'              : [f'PC{i+1}' for i in range(len(eigenvalues))],
        'Eigenvalue'             : np.round(eigenvalues, 4),
        'Variance Explained (%)'  : np.round(pca_full.explained_variance_ratio_ * 100, 2),
        'Cumulative Variance (%)' : np.round(np.cumsum(pca_full.explained_variance_ratio_ * 100), 2),
    })
    eigen_df.to_excel(writer, sheet_name='Eigenvalues', index=False)
    apcs_df = pd.DataFrame(APCS, columns=[f'APCS_{i+1}' for i in range(n_factors)])
    apcs_df.insert(0, 'Sample', df['Sample'].values)
    apcs_df.to_excel(writer, sheet_name='APCS Scores', index=False)
    coeff_df.round(4).to_excel(writer, sheet_name='MLR Coefficients')
    pct_df.round(2).to_excel(writer, sheet_name='% Contributions')
print("✓ APCS_MLR_results.xlsx saved")

✓ sankey_APCS_MLR.html saved
✓ APCS_MLR_results.xlsx saved
